# ML Pipeline 5: Donor Impact Allocation Model
Dual-model pipeline (predictive + explanatory) for estimated donation impact allocation.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.db import load_env, build_engine
from src.features import build_frame, split_xy
from src.modeling import train_predictive_top_area, train_explanatory_share, donor_impact_output


## Problem Framing
- Predictive objective: predict top allocation program area for a donation.
- Explanatory objective: estimate which features are associated with higher education allocation share.
- Stakeholder: donor relations and fundraising operations teams.

## Business KPI Mapping
- Decision enabled: donor impact estimate display and targeting narrative.
- Primary KPI: top-program prediction quality (macro F1) and allocation-share fit (R2).
- Guardrails: avoid overclaiming causal outcomes; maintain estimate stability across periods.
- Success criteria: non-trivial predictive lift and interpretable driver consistency.

In [ ]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_frame(engine)
X, y_top, y_share_education = split_xy(df)
print('Rows:', len(df), '| Distinct donors:', df['supporter_id'].nunique())
df.head()

In [ ]:
# Data Understanding Audit
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))
audit = {
    'rows': len(df),
    'program_area_classes': int(y_top.nunique()),
    'zero_donation_value_rows': int((df['donation_value'] <= 0).sum()),
}
print('Audit summary:', audit)
print('Feature rationale: donation/supporter context explains likely allocation patterns observed in history.')

In [ ]:
pred_metrics, best_top_model = train_predictive_top_area(X, y_top)
print('Predictive metrics (top program area):')
display(pred_metrics)

In [ ]:
expl_metrics, linear_share_model, rf_share_model, coef_df = train_explanatory_share(X, y_share_education)
print('Share model metrics:')
display(expl_metrics)
print('Top positive explanatory drivers (education share):')
display(coef_df.head(12))
print('Top negative explanatory drivers:')
display(coef_df.tail(12).sort_values('coefficient'))

In [ ]:
impact_out = donor_impact_output(df, best_top_model, linear_share_model)
print('Donor impact output sample:')
display(impact_out.head(20))

## Evaluation in Business Terms
- Misclassifying top area can mislead donor messaging.
- Over/under-estimating allocation shares can distort expectation setting.
- Use outputs as impact estimates based on historical patterns, not guaranteed causal outcomes.

## Operationalization Policy + Monitoring
- Prediction policy: display top-program estimate + education-share range.
- Action bands: high education-share estimates for education-focused messaging.
- Retraining cadence: monthly or when monitored quality drops.
- References:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Deployment Notes
Integrate as donor profile panel: input donation context, output estimated allocation profile and uncertainty range.